In [ ]:
# Cell 1 — Install dependencies
!pip install -q gradio ultralytics transformers xgboost opencv-python-headless supervision
print('\u2705 Dependencies installed')

In [ ]:
# Cell 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('\u2705 Drive mounted')

In [ ]:
# Cell 3 — Imports
import os
import cv2
import torch
import pickle
import warnings
import tempfile
import numpy as np
import torch.nn as nn
from collections import defaultdict
import scipy.stats as scipy_stats
from transformers import VideoMAEModel, VideoMAEImageProcessor
from ultralytics import YOLO
import gradio as gr

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# Cell 4 — Paths and constants
DEMO_DIR       = '/content/drive/MyDrive/L6/FYP/SmartRef_Thesis_w1953520_20220707_Akthar_Rasheed/DemoSystem'
MODEL_PATH     = f'{DEMO_DIR}/smartref_best.pt'
XGB_PATH       = f'{DEMO_DIR}/xgb_enhanced_model.pkl'
YOLO_POSE_PATH = f'{DEMO_DIR}/yolo26x-pose.pt'
YOLO_BALL_PATH = f'{DEMO_DIR}/yolo26x.pt'
VIDEOMAE_NAME  = 'MCG-NJU/videomae-base-finetuned-kinetics'

NN_WEIGHT  = 0.75
XGB_WEIGHT = 0.25
THRESHOLD  = 0.460
NUM_FRAMES = 16
IMG_SIZE   = 224
POSE_DIM   = 20

for name, path in [
    ('smartref_best.pt',       MODEL_PATH),
    ('xgb_enhanced_model.pkl', XGB_PATH),
    ('yolo26x-pose.pt',        YOLO_POSE_PATH),
    ('yolo26x.pt',             YOLO_BALL_PATH),
]:
    status = '\u2705' if os.path.exists(path) else '\u274c MISSING'
    print(f'{status}  {name}')

In [ ]:
# ── CELL 5: SmartRefNet Architecture ────────────────────────────────────────
# Architecture reconstructed exactly from checkpoint keys + shapes:
#   video_proj : LayerNorm(768) → Linear(768, 128)
#   pose_gru   : BiGRU(input=20, hidden=64, layers=2)  → output 128
#   pose_proj  : LayerNorm(128) → Linear(128, 128)
#   fusion     : Linear(256,256) → BN(256) → ReLU → Dropout
#                → Linear(256,128) → ReLU → Dropout → Linear(128,2)
class SmartRefNet(nn.Module):
    def __init__(self, videomae_model_name, pose_input_dim=20,
                 gru_hidden=64, proj_dim=128, dropout=0.3):
        super().__init__()
        self.videomae = VideoMAEModel.from_pretrained(videomae_model_name)
        video_hidden = self.videomae.config.hidden_size  # 768

        self.video_proj = nn.Sequential(
            nn.LayerNorm(video_hidden),
            nn.Linear(video_hidden, proj_dim)
        )
        self.pose_gru = nn.GRU(
            pose_input_dim, gru_hidden,
            num_layers=2, batch_first=True,
            bidirectional=True, dropout=dropout
        )
        self.pose_proj = nn.Sequential(
            nn.LayerNorm(gru_hidden * 2),
            nn.Linear(gru_hidden * 2, proj_dim)
        )
        self.fusion = nn.Sequential(
            nn.Linear(proj_dim * 2, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 2)
        )

    def forward(self, pixel_values, pose_seq):
        video_out  = self.videomae(pixel_values=pixel_values)
        video_feat = self.video_proj(video_out.last_hidden_state[:, 0, :])
        pose_out, _ = self.pose_gru(pose_seq)
        pose_feat  = self.pose_proj(pose_out.mean(dim=1))
        fused  = torch.cat([video_feat, pose_feat], dim=-1)
        logits = self.fusion(fused)  # (batch, 2)
        return logits

print('✅ SmartRefNet class defined')

In [ ]:
# Cell 6 — Feature extraction functions (matching training notebook exactly)
# 20 features: 8 spatial + 8 validity masks + velocity + acceleration + 2 ball = 20
# Feature index map (must match training):
#   0  torso_distance          (body-scale normalised)
#   1  wristA_to_torsoB        (contact proxy A→B)
#   2  wristB_to_torsoA        (contact proxy B→A)
#   3  left_knee_angle_A       (normalised /180)
#   4  right_knee_angle_A
#   5  left_knee_angle_B
#   6  right_knee_angle_B
#   7  foot_min_distance
#   8  valid_torso             (validity masks 8-15)
#   9  valid_wristA
#  10  valid_wristB
#  11  valid_lkneeA
#  12  valid_rkneeA
#  13  valid_lkneeB
#  14  valid_rkneeB
#  15  valid_feet
#  16  velocity                (frame-to-frame torso_distance delta)
#  17  acceleration            (frame-to-frame velocity delta)
#  18  ball_to_wrist_dist
#  19  possession_proxy

import math as _math

KP_CONF_THRESH = 0.3

# COCO keypoint indices
KP_L_SHOULDER = 5;  KP_R_SHOULDER = 6
KP_L_ELBOW    = 7;  KP_R_ELBOW    = 8
KP_L_WRIST    = 9;  KP_R_WRIST    = 10
KP_L_HIP      = 11; KP_R_HIP      = 12
KP_L_KNEE     = 13; KP_R_KNEE     = 14
KP_L_ANKLE    = 15; KP_R_ANKLE    = 16


def _safe_kp(kpts, idx):
    """Return (x, y) if confidence > threshold, else None. kpts is (17,3) array."""
    if kpts is None or len(kpts) <= idx:
        return None
    x, y, c = float(kpts[idx][0]), float(kpts[idx][1]), float(kpts[idx][2])
    return (x, y) if c > KP_CONF_THRESH else None


def _kp_dist(p1, p2):
    if p1 is None or p2 is None:
        return 0.0
    return float(_math.hypot(p1[0] - p2[0], p1[1] - p2[1]))


def _midpoint(p1, p2):
    if p1 is None or p2 is None:
        return p1 or p2
    return ((p1[0] + p2[0]) / 2.0, (p1[1] + p2[1]) / 2.0)


def _torso_center(kpts):
    """Hip midpoint as torso centre-of-mass proxy."""
    lh = _safe_kp(kpts, KP_L_HIP)
    rh = _safe_kp(kpts, KP_R_HIP)
    return _midpoint(lh, rh)


def _body_scale(kpts):
    """Shoulder-to-shoulder distance as body scale reference. Fallback 1.0."""
    ls = _safe_kp(kpts, KP_L_SHOULDER)
    rs = _safe_kp(kpts, KP_R_SHOULDER)
    if ls is not None and rs is not None:
        return max(1.0, float(_math.hypot(ls[0]-rs[0], ls[1]-rs[1])))
    return 1.0


def _angle_3pts(a, b, c):
    """Angle at vertex b in degrees. Returns (angle, valid_flag)."""
    if a is None or b is None or c is None:
        return 0.0, 0
    va = (a[0]-b[0], a[1]-b[1])
    vc = (c[0]-b[0], c[1]-b[1])
    na = _math.hypot(*va); nc = _math.hypot(*vc)
    if na < 1e-9 or nc < 1e-9:
        return 0.0, 0
    cos_a = max(-1.0, min(1.0, (va[0]*vc[0] + va[1]*vc[1]) / (na * nc)))
    return float(_math.degrees(_math.acos(cos_a))), 1


def richer_frame_feats(kp_a, kp_b):
    """
    Compute 8 spatial features + 8 validity masks from two players' keypoints.
    kp_a, kp_b: (17, 3) numpy arrays or None.
    Returns (feats np.array(8,), mask np.array(8,))
    """
    scale_a = _body_scale(kp_a)
    scale_b = _body_scale(kp_b)
    scale   = max(1.0, (scale_a + scale_b) / 2.0)

    ta = _torso_center(kp_a)
    tb = _torso_center(kp_b)

    lwa = _safe_kp(kp_a, KP_L_WRIST); rwa = _safe_kp(kp_a, KP_R_WRIST)
    lwb = _safe_kp(kp_b, KP_L_WRIST); rwb = _safe_kp(kp_b, KP_R_WRIST)

    lha = _safe_kp(kp_a, KP_L_HIP);   rha = _safe_kp(kp_a, KP_R_HIP)
    lka = _safe_kp(kp_a, KP_L_KNEE);  rka = _safe_kp(kp_a, KP_R_KNEE)
    laa = _safe_kp(kp_a, KP_L_ANKLE); raa = _safe_kp(kp_a, KP_R_ANKLE)

    lhb = _safe_kp(kp_b, KP_L_HIP);   rhb = _safe_kp(kp_b, KP_R_HIP)
    lkb = _safe_kp(kp_b, KP_L_KNEE);  rkb = _safe_kp(kp_b, KP_R_KNEE)
    lab = _safe_kp(kp_b, KP_L_ANKLE); rab = _safe_kp(kp_b, KP_R_ANKLE)

    # Feature 0: torso distance
    f0 = (_kp_dist(ta, tb) / scale) if (ta and tb) else 0.0
    v0 = int(ta is not None and tb is not None)

    # Feature 1: min wristA to torsoB
    wA_tb = [_kp_dist(w, tb)/scale for w in [lwa, rwa] if w and tb]
    f1 = min(wA_tb) if wA_tb else 0.0
    v1 = int(bool(wA_tb))

    # Feature 2: min wristB to torsoA
    wB_ta = [_kp_dist(w, ta)/scale for w in [lwb, rwb] if w and ta]
    f2 = min(wB_ta) if wB_ta else 0.0
    v2 = int(bool(wB_ta))

    # Features 3-6: knee angles normalised /180
    f3, v3 = _angle_3pts(lha, lka, laa); f3 /= 180.0
    f4, v4 = _angle_3pts(rha, rka, raa); f4 /= 180.0
    f5, v5 = _angle_3pts(lhb, lkb, lab); f5 /= 180.0
    f6, v6 = _angle_3pts(rhb, rkb, rab); f6 /= 180.0

    # Feature 7: minimum foot distance
    feet_a = [p for p in [laa, raa] if p]
    feet_b = [p for p in [lab, rab] if p]
    if feet_a and feet_b:
        f7 = min(_kp_dist(fa, fb)/scale for fa in feet_a for fb in feet_b)
        v7 = 1
    else:
        f7, v7 = 0.0, 0

    feats = np.array([f0, f1, f2, f3, f4, f5, f6, f7], dtype=np.float32)
    mask  = np.array([v0, v1, v2, v3, v4, v5, v6, v7], dtype=np.float32)
    return feats, mask


def build_pose_sequence(off_kps_list, def_kps_list, ball_xy_list=None):
    """
    Build (T, 20) pose feature array from per-frame keypoint lists.
    off_kps_list, def_kps_list: list of (17,3) arrays.
    ball_xy_list: list of (cx, cy) or None per frame.
    """
    T = len(off_kps_list)
    pose_seq = np.zeros((T, 20), dtype=np.float32)
    prev_td  = None
    prev_vel = 0.0

    for i, (kp_a, kp_b) in enumerate(zip(off_kps_list, def_kps_list)):
        feats, mask = richer_frame_feats(kp_a, kp_b)

        torso_d = feats[0]
        vel = 0.0 if prev_td is None else (torso_d - prev_td)
        acc = vel - prev_vel if prev_td is not None else 0.0
        prev_td  = torso_d
        prev_vel = vel

        # Ball features
        ball_xy = ball_xy_list[i] if ball_xy_list else None
        if ball_xy is not None and kp_a is not None:
            scale_a = _body_scale(kp_a)
            bx, by  = float(ball_xy[0]), float(ball_xy[1])
            lw = _safe_kp(kp_a, KP_L_WRIST)
            rw = _safe_kp(kp_a, KP_R_WRIST)
            dists = []
            if lw: dists.append(_math.hypot(lw[0]-bx, lw[1]-by) / max(scale_a, 1.0))
            if rw: dists.append(_math.hypot(rw[0]-bx, rw[1]-by) / max(scale_a, 1.0))
            if dists:
                btw  = min(dists)
                poss = 1.0 if btw < 1.5 else 0.0
            else:
                btw, poss = 0.0, 0.0
        else:
            btw, poss = 0.0, 0.0

        pose_seq[i] = np.concatenate([
            feats,
            mask,
            np.array([vel, acc, btw, poss], dtype=np.float32)
        ])

    return pose_seq


def build_xgb_features(pose_seq_np):
    """
    (T, 20) pose sequence -> 150-dim float32 feature vector.
    7 stats x 20 features = 140, plus 10 contact dynamics = 150.
    Stats: mean, std, min, max, delta(last-first), argmin_t, argmax_t
    """
    T = len(pose_seq_np)
    feats = []

    for j in range(20):
        col = pose_seq_np[:, j].astype(float)
        feats.extend([
            float(np.mean(col)),
            float(np.std(col)),
            float(np.min(col)),
            float(np.max(col)),
            float(col[-1] - col[0]),                              # delta
            float(np.argmin(col)) / max(T - 1, 1),               # argmin_t
            float(np.argmax(col)) / max(T - 1, 1),               # argmax_t
        ])

    # Contact dynamics (features matching training notebook exactly)
    torso_d  = pose_seq_np[:, 0].astype(float)
    velocity = pose_seq_np[:, 16].astype(float)
    accel    = pose_seq_np[:, 17].astype(float)
    wristA   = pose_seq_np[:, 1].astype(float)
    wristB   = pose_seq_np[:, 2].astype(float)

    close_mask = torso_d < 0.3
    n_close    = float(close_mask.sum())

    max_consec = cur = 0
    for v in close_mask:
        cur = cur + 1 if v else 0
        max_consec = max(max_consec, cur)

    min_td       = float(torso_d.min())
    min_idx      = int(np.argmin(torso_d))
    min_td_frame = float(min_idx) / max(T - 1, 1)
    vel_at_min   = float(velocity[min_idx])
    acc_at_min   = float(accel[min_idx])
    max_abs_acc  = float(np.abs(accel).max())
    mean_close_vel = float(velocity[close_mask].mean()) if close_mask.any() else 0.0

    t_axis = np.arange(T, dtype=np.float32)
    slope  = float(np.polyfit(t_axis, torso_d, 1)[0]) if torso_d.std() > 1e-6 else 0.0

    contact_asym = float(abs(wristA[min_idx] - wristB[min_idx]))

    feats.extend([
        n_close, float(max_consec), min_td, min_td_frame,
        vel_at_min, acc_at_min, max_abs_acc,
        mean_close_vel, slope, contact_asym
    ])

    arr = np.array(feats, dtype=np.float32)
    assert len(arr) == 150, f'Expected 150 XGB features, got {len(arr)}'
    return np.nan_to_num(arr)


print('✅ Feature extraction functions defined (matching training notebook)')

In [ ]:
# Cell 7 — Load all models
print('Loading YOLO26x-Pose...')
yolo_pose = YOLO(YOLO_POSE_PATH)
print('  \u2705 YOLO26x-Pose loaded')

print('Loading YOLO26x (ball detection)...')
yolo_ball = YOLO(YOLO_BALL_PATH)
print('  \u2705 YOLO26x loaded')

print('Loading VideoMAE processor...')
processor = VideoMAEImageProcessor.from_pretrained(VIDEOMAE_NAME)
print('  \u2705 VideoMAE processor loaded')

print('Loading SmartRefNet weights...')
nn_model = SmartRefNet(VIDEOMAE_NAME).to(DEVICE)
ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
    nn_model.load_state_dict(ckpt['model_state_dict'])
else:
    nn_model.load_state_dict(ckpt)
nn_model.eval()
print('  \u2705 SmartRefNet loaded')

print('Loading Enhanced Temporal XGBoost...')
with open(XGB_PATH, 'rb') as f:
    xgb_model = pickle.load(f)
print('  \u2705 XGBoost loaded')

print('\n\u2705 All models loaded successfully')

In [ ]:
# Cell 8 — Inference pipeline

def extract_frames_uniform(video_path, n=NUM_FRAMES):
    """Extract n uniformly spaced frames. Returns list of BGR frames."""
    cap   = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        total = 1
    indices = np.linspace(0, max(0, total - 1), n, dtype=int)
    frames  = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if ret:
            frames.append(frame)
        elif frames:
            frames.append(frames[-1].copy())
        else:
            frames.append(np.zeros((224, 224, 3), dtype=np.uint8))
    cap.release()
    return frames


def detect_interaction_pair(frames):
    """
    Track players across frames with YOLO26x-Pose + ByteTrack.
    Find the pair of track IDs that co-appear most often.
    Returns (off_kps, def_kps, off_bbs, def_bbs) — one entry per frame.
    """
    frame_data = []  # [{track_id: (kps_array, bbox_xyxy)}, ...]
    co_appear  = defaultdict(int)

    for frame in frames:
        res_list = yolo_pose.track(frame, persist=True, conf=0.15,
                                   classes=[0], verbose=False)
        fd = {}
        if res_list and res_list[0].boxes is not None:
            res   = res_list[0]
            boxes = res.boxes
            kps   = res.keypoints
            for i in range(len(boxes)):
                tid  = int(boxes.id[i].item()) if boxes.id is not None else i
                xyxy = boxes.xyxy[i].cpu().numpy().tolist()
                kp   = (kps.data[i].cpu().numpy()
                        if kps is not None else np.zeros((17, 3)))
                fd[tid] = (kp, xyxy)
        frame_data.append(fd)
        tids = list(fd.keys())
        for a in range(len(tids)):
            for b in range(a + 1, len(tids)):
                co_appear[tuple(sorted([tids[a], tids[b]]))] += 1

    if not co_appear:
        return None, None, None, None

    id_off, id_def = max(co_appear, key=co_appear.get)
    dummy_kp = np.zeros((17, 3))
    dummy_bb = [0, 0, 100, 200]

    off_kps, def_kps, off_bbs, def_bbs = [], [], [], []
    for fd in frame_data:
        by_area = sorted(fd.values(),
                         key=lambda x: (x[1][2]-x[1][0])*(x[1][3]-x[1][1]),
                         reverse=True)
        # Offender
        if id_off in fd:
            kp, bb = fd[id_off]
        elif len(by_area) >= 1:
            kp, bb = by_area[0]
        else:
            kp, bb = dummy_kp, dummy_bb
        off_kps.append(kp); off_bbs.append(bb)
        # Defender
        if id_def in fd:
            kp, bb = fd[id_def]
        elif len(by_area) >= 2:
            kp, bb = by_area[1]
        else:
            kp, bb = dummy_kp, dummy_bb
        def_kps.append(kp); def_bbs.append(bb)

    return off_kps, def_kps, off_bbs, def_bbs


def build_interaction_crop(frames, off_bboxes, def_bboxes, margin=0.15):
    """
    For each frame: compute union bbox of both players, add margin,
    crop and convert to RGB. Returns list of RGB crops.
    """
    crops = []
    for frame, bb_o, bb_d in zip(frames, off_bboxes, def_bboxes):
        H, W = frame.shape[:2]
        x1   = min(bb_o[0], bb_d[0]);  y1 = min(bb_o[1], bb_d[1])
        x2   = max(bb_o[2], bb_d[2]);  y2 = max(bb_o[3], bb_d[3])
        mw   = (x2 - x1) * margin;     mh = (y2 - y1) * margin
        x1   = max(0, int(x1 - mw));   y1 = max(0, int(y1 - mh))
        x2   = min(W, int(x2 + mw));   y2 = min(H, int(y2 + mh))
        crop = frame[y1:y2, x1:x2] if x2 > x1 and y2 > y1 else frame
        crops.append(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    return crops


def smartref_predict(video_path):
    """Full SmartRef-Net inference. Returns (label, ensemble_prob, nn_prob, xgb_prob)."""
    global nn_model, xgb_model, yolo_pose, yolo_ball, processor

    # 1. Extract frames
    frames = extract_frames_uniform(video_path, NUM_FRAMES)
    W = frames[0].shape[1] if frames else 1280
    H = frames[0].shape[0] if frames else 720

    # 2. Detect interaction pair
    off_kps, def_kps, off_bbs, def_bbs = detect_interaction_pair(frames)
    if off_kps is None:
        off_kps = [np.zeros((17, 3))] * len(frames)
        def_kps = [np.zeros((17, 3))] * len(frames)
        off_bbs = [[0,   0, 100, 200]] * len(frames)
        def_bbs = [[110, 0, 210, 200]] * len(frames)

    # 3. Build pose feature sequence (T x 20) — matches training notebook feature layout
    pose_seq_np = build_pose_sequence(off_kps, def_kps, ball_xy_list=None)

    # 4. Build interaction crops -> pixel_values
    crops = build_interaction_crop(frames, off_bbs, def_bbs)
    inputs = processor(crops, return_tensors='pt')
    pixel_values = inputs['pixel_values'].to(DEVICE)
    # Force correct shape (1, T, C, H, W) — processor may return extra dims
    pixel_values = pixel_values.contiguous().view(1, NUM_FRAMES, 3, IMG_SIZE, IMG_SIZE)

    # 5. NN forward pass — output is (1, 2) logits, class 1 = FOUL
    pose_tensor = torch.tensor(pose_seq_np).unsqueeze(0).to(DEVICE)  # (1, T, 20)
    with torch.no_grad():
        logits = nn_model(pixel_values, pose_tensor)          # (1, 2)
        nn_prob = float(torch.softmax(logits, dim=-1)[0, 1].cpu().item())

    # 6. XGBoost prediction
    xgb_feats = build_xgb_features(pose_seq_np).reshape(1, -1)
    xgb_feats = np.nan_to_num(xgb_feats)
    xgb_prob  = float(xgb_model.predict_proba(xgb_feats)[0][1])

    # 7. Ensemble
    ensemble_prob = NN_WEIGHT * nn_prob + XGB_WEIGHT * xgb_prob
    label = 'FOUL' if ensemble_prob >= THRESHOLD else 'NO FOUL'

    return label, ensemble_prob, nn_prob, xgb_prob


print('\u2705 Inference pipeline defined')

In [ ]:
# Cell 9 — Gradio interface

def gradio_predict(video_file):
    if video_file is None:
        return ('\u26a0\ufe0f No video uploaded', 'Please upload an .mp4 clip', '')
    label, ensemble_prob, nn_prob, xgb_prob = smartref_predict(video_file)
    verdict    = '\U0001f6a8 FOUL' if label == 'FOUL' else '\u2705 NO FOUL'
    confidence = (
        f'Ensemble:           {ensemble_prob * 100:.1f}%\n'
        f'NN Component:       {nn_prob * 100:.1f}%\n'
        f'XGBoost Component:  {xgb_prob * 100:.1f}%'
    )
    model_info = (
        f'Threshold: {THRESHOLD}\n'
        f'NN weight: {NN_WEIGHT}  |  XGBoost weight: {XGB_WEIGHT}'
    )
    return verdict, confidence, model_info


with gr.Blocks(theme=gr.themes.Soft(), title='SmartRef-Net Demo') as demo:
    gr.Markdown("""
    # SmartRef-Net \u2014 AI Basketball Foul Detection
    **Student:** Muhammed Akthar Abdul Rasheed W1953520
    """)

    with gr.Row():
        with gr.Column():
            video_input = gr.Video(label='Upload Basketball Clip (.mp4)')
            run_btn     = gr.Button('Analyse Clip', variant='primary')
        with gr.Column():
            verdict_out    = gr.Textbox(label='Verdict',     interactive=False)
            confidence_out = gr.Textbox(label='Confidence',  interactive=False, lines=3)
            model_info_out = gr.Textbox(label='Model Info',  interactive=False, lines=2)

    run_btn.click(
        fn=gradio_predict,
        inputs=[video_input],
        outputs=[verdict_out, confidence_out, model_info_out],
    )

    gr.Markdown("""
    ---
    **Architecture:** VideoMAE + Pose BiGRU (NN = 75%) + Enhanced Temporal XGBoost (25%)
    **Dataset:** 275 clips  |  **External test F1:** 0.6111
    """)

# The public gradio.live URL will appear below — share with supervisor
demo.launch(share=True, debug=False)